# Qwen2-VL (modernized KerasHub) — Colab eval

Run on an **A100** runtime (`Runtime → Change runtime type → A100`).

This notebook:
1. Clones the `qwen2-vl-modernized` branch and installs it.
2. Runs the qwen2_vl unit tests.
3. Loads `Qwen/Qwen2-VL-2B-Instruct` in HF and via `hf://` in KerasHub.
4. Checks numerical parity (vision tower + full logits + position_ids).
5. Runs generation on image, multi-image, and video inputs.
6. Benchmarks decode speed + vision-encoder latency (KerasHub vs HF).

In [ ]:
# @title 0. Setup — clone branch + install
import os

os.environ["KERAS_BACKEND"] = "tensorflow"  # must precede keras import

BRANCH = "qwen2-vl-modernized"
REPO = "https://github.com/samudraneel05/keras-hub.git"

!git clone -b $BRANCH $REPO /content/keras-hub
%cd /content/keras-hub
!git pull origin $BRANCH  # pick up any newer pushes

!pip install -e . --quiet
!pip install -U "transformers>=4.51" safetensors accelerate requests \
    --quiet
# Colab ships a stale PIL mix; force-reinstall fixes the _Ink import error
!pip install -U --force-reinstall --no-cache-dir pillow --quiet

In [ ]:
import time

import keras
import numpy as np
import tensorflow as tf
import torch
import transformers

import keras_hub

print(
    "keras_hub",
    keras_hub.__version__,
    "| tf",
    tf.__version__,
    "| torch",
    torch.__version__,
    "| hf",
    transformers.__version__,
)
print("TF GPUs :", tf.config.list_physical_devices("GPU"))
print(
    "Torch   :",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "NO GPU — switch to an A100 runtime!",
)

## 1. Unit tests
Backbone, vision encoder, preprocessor, tokenizer, causal LM, image converter.

In [ ]:
!python -m pytest keras_hub/src/models/qwen2_vl/ -x -q --no-header

## 2. Test assets
Two real images (Qwen demo + a HF doc image) and a synthetic 4-frame video.

In [ ]:
import io

import requests
from PIL import Image


def load_image(url, fallback_size=(448, 448)):
    try:
        return Image.open(
            io.BytesIO(requests.get(url, timeout=30).content)
        ).convert("RGB")
    except Exception as e:
        print("download failed, using synthetic image:", e)
        rng = np.random.default_rng(0)
        return Image.fromarray(
            rng.integers(0, 255, (*fallback_size, 3), dtype=np.uint8)
        )


# demo.jpeg is the canonical Qwen2-VL example image; bee.jpg is a standard
# transformers doc image.
img1 = load_image(
    "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
)
img2 = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/"
    "resolve/main/bee.jpg",
    fallback_size=(224, 224),
)
img1

In [ ]:
# Synthetic 4-frame video (moving square) for pipeline testing.
rng = np.random.default_rng(1)
video = rng.integers(0, 60, (4, 224, 224, 3), dtype=np.uint8)
for f in range(4):
    video[f, 40 + f * 20 : 100 + f * 20, 60:160] = [255, 255, 255]
print("video:", video.shape)

## 3. Load models
HF (fp32 for tight parity) and KerasHub via `hf://` preset conversion.

In [ ]:
from transformers import AutoProcessor
from transformers import Qwen2VLForConditionalGeneration

HF_ID = "Qwen/Qwen2-VL-2B-Instruct"

hf_processor = AutoProcessor.from_pretrained(HF_ID)
hf_model = Qwen2VLForConditionalGeneration.from_pretrained(
    HF_ID, torch_dtype=torch.float32, device_map="cuda"
).eval()
n_hf = sum(p.numel() for p in hf_model.parameters())
print("HF params: %.2fB" % (n_hf / 1e9))

In [ ]:
# This exercises the full conversion path: convert_backbone_config,
# convert_weights, convert_tokenizer, load_image_converter_config.
keras_lm = keras_hub.models.Qwen2VLCausalLM.from_preset(f"hf://{HF_ID}")
keras_lm.compile(sampler="greedy")
print("KerasHub params: %.2fB" % (keras_lm.count_params() / 1e9))

## 4. Numerical parity
Same image + prompt through both stacks; compare logits on real tokens.

In [ ]:
PROMPT_IMG = (
    "<|im_start|>user\n"
    "<|vision_start|><|image_pad|><|vision_end|>"
    "Describe this image.<|im_end|>\n"
    "<|im_start|>assistant\n"
)

hf_inputs = hf_processor(
    text=[PROMPT_IMG], images=[img1], return_tensors="pt"
).to("cuda")
with torch.no_grad():
    hf_logits = hf_model(**hf_inputs).logits.float().cpu().numpy()

k_in = keras_lm.preprocessor.generate_preprocess(
    {"prompts": [PROMPT_IMG], "images": [np.asarray(img1)]}
)
print({k: tuple(v.shape) for k, v in k_in.items()})

k_logits = keras.ops.convert_to_numpy(keras_lm(k_in))

n_real = int(np.asarray(k_in["padding_mask"]).sum())
diff = np.abs(k_logits[0, :n_real] - hf_logits[0, :n_real])
print(
    f"logits max diff {diff.max():.3e}  mean {diff.mean():.3e} "
    f"over {n_real} positions"
)

In [ ]:
# Vision-tower-only parity.
pv_hf = hf_inputs["pixel_values"]
grid = hf_inputs["image_grid_thw"]
with torch.no_grad():
    hf_ve = hf_model.model.visual(pixel_values=pv_hf, grid_thw=grid)
hf_ve = getattr(hf_ve, "pooler_output", hf_ve)
hf_ve = hf_ve.float().cpu().numpy()

# HF patch flat layout (C, t, ph, pw) → keras (t, ph, pw, C).
c = 3
tps = keras_lm.backbone.vision_encoder.temporal_patch_size
ps = keras_lm.backbone.vision_encoder.patch_size
pv_k = pv_hf.cpu().numpy().reshape(-1, c, tps, ps, ps)
pv_k = np.transpose(pv_k, (0, 2, 3, 4, 1))
k_ve = keras.ops.convert_to_numpy(
    keras_lm.backbone.vision_encoder(
        keras.ops.convert_to_tensor(pv_k),
        keras.ops.convert_to_tensor(grid.cpu().numpy(), dtype="int32"),
    )
)
d = np.abs(k_ve - hf_ve)
print(f"vision tower max diff {d.max():.3e}  mean {d.mean():.3e}")

## 5. Generation — single image, multi-image, video
Greedy decoding on both stacks; outputs should match or be near-identical.

In [ ]:
def hf_generate(text, images=None, videos=None, max_new_tokens=64):
    inputs = hf_processor(
        text=[text], images=images, videos=videos, return_tensors="pt"
    ).to("cuda")
    t0 = time.time()
    with torch.no_grad():
        out = hf_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False
        )
    dt = time.time() - t0
    gen = out[0][inputs["input_ids"].shape[1] :]
    return hf_processor.decode(gen, skip_special_tokens=True), gen.numel() / dt


def keras_generate(text, images=None, videos=None, max_new_tokens=64):
    payload = {"prompts": text}
    if images is not None:
        payload["images"] = (
            np.asarray(images[0])
            if len(images) == 1
            else [np.asarray(i) for i in images]
        )
    if videos is not None:
        payload["videos"] = (
            np.asarray(videos[0])
            if len(videos) == 1
            else [np.asarray(v) for v in videos]
        )
    pp = keras_lm.preprocessor.generate_preprocess(payload)
    prompt_len = int(np.asarray(pp["padding_mask"]).sum())
    t0 = time.time()
    out = keras_lm.generate(
        payload, max_length=prompt_len + max_new_tokens, strip_prompt=True
    )
    dt = time.time() - t0
    n_new = len(keras_lm.preprocessor.tokenizer(out))
    return out, n_new / dt


hf_text, hf_tps = hf_generate(PROMPT_IMG, images=[img1])
k_text, k_tps = keras_generate(PROMPT_IMG, images=[img1])
print("HF   :", repr(hf_text))
print("KERAS:", repr(k_text))
print(f"HF {hf_tps:.1f} tok/s  |  Keras {k_tps:.1f} tok/s (incl. prefill)")

In [ ]:
# Multi-image.
PROMPT_2IMG = (
    "<|im_start|>user\n"
    "<|vision_start|><|image_pad|><|vision_end|>"
    "<|vision_start|><|image_pad|><|vision_end|>"
    "What do these two images have in common?<|im_end|>\n"
    "<|im_start|>assistant\n"
)
hf_text, _ = hf_generate(PROMPT_2IMG, images=[img1, img2])
k_text, _ = keras_generate(PROMPT_2IMG, images=[img1, img2])
print("HF   :", repr(hf_text))
print("KERAS:", repr(k_text))

In [ ]:
# Video (synthetic frames — exercises temporal grid + per-frame attention).
PROMPT_VID = (
    "<|im_start|>user\n"
    "<|vision_start|><|video_pad|><|vision_end|>"
    "What happens in this video?<|im_end|>\n"
    "<|im_start|>assistant\n"
)
hf_text, _ = hf_generate(PROMPT_VID, videos=[video])
k_text, _ = keras_generate(PROMPT_VID, videos=[video])
print("HF   :", repr(hf_text))
print("KERAS:", repr(k_text))

## 6. Speed benchmark
Decode throughput with a fixed 128 new tokens (no early stop), batch 1.
Compare against HF's published ~35 tok/s for 2B BF16 on A100 (this run is fp32,
so expect somewhat lower on both sides — the comparison is apples-to-apples).

In [ ]:
def bench_hf(text, images, n=128, warmup=1, reps=3):
    inputs = hf_processor(text=[text], images=images, return_tensors="pt").to(
        "cuda"
    )
    for _ in range(warmup):
        hf_model.generate(
            **inputs, max_new_tokens=n, min_new_tokens=n, do_sample=False
        )
    ts = []
    for _ in range(reps):
        t0 = time.time()
        hf_model.generate(
            **inputs, max_new_tokens=n, min_new_tokens=n, do_sample=False
        )
        torch.cuda.synchronize()
        ts.append(n / (time.time() - t0))
    return np.mean(ts)


def bench_keras(text, images, n=128, warmup=1, reps=3):
    payload = {"prompts": text, "images": [np.asarray(i) for i in images]}
    pp = keras_lm.preprocessor.generate_preprocess(payload)
    plen = int(np.asarray(pp["padding_mask"]).sum())
    for _ in range(warmup):
        keras_lm.generate(payload, max_length=plen + n, stop_token_ids=None)
    ts = []
    for _ in range(reps):
        t0 = time.time()
        keras_lm.generate(payload, max_length=plen + n, stop_token_ids=None)
        ts.append(n / (time.time() - t0))
    return np.mean(ts)


hf_tps = bench_hf(PROMPT_IMG, [img1])
k_tps = bench_keras(PROMPT_IMG, [img1])
print(f"HF 2B fp32 : {hf_tps:.1f} tok/s")
print(f"KerasHub   : {k_tps:.1f} tok/s")

In [ ]:
# Vision encoder latency (single 224x224 image → 256 patches).
ve = keras_lm.backbone.vision_encoder
t0 = time.time()
for _ in range(10):
    ve(
        keras.ops.convert_to_tensor(pv_k),
        keras.ops.convert_to_tensor(grid.cpu().numpy(), dtype="int32"),
    )
print(
    f"vision tower: {(time.time() - t0) / 10 * 1000:.1f} ms/image "
    f"({pv_k.shape[0]} patches)"
)

## 7. Notes
- For full accuracy benchmarking (DocVQA, MMMU, Video-MME), wire the
  converted model into [VLMEvalKit](https://github.com/open-compass/VLMEvalKit)
  — logits-level parity (~1e-3 with fp32/bf16 tolerance) means scores should
  track HF's published numbers.
- Known perf headroom: vision attention uses a block-diagonal mask
  (O(S²)); HF chunks per frame. Fine for typical images, heavy at the
  12.8M-pixel cap.